In [1]:
import torch
import torchvision

In [2]:
def download_cifar10():
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
    return trainset, testset

In [3]:
def flatten_images_3d(dataset: torch.utils.data.Dataset):
    flattened_images = []
    for image, _ in dataset:
        images_flat = image.view(-1)
        flattened_images.append(images_flat)
    return torch.stack(flattened_images)

In [4]:
def flatten_images_2d(dataset: torch.utils.data.Dataset): 
    flattened_images = []
    for image, _ in dataset:
        image_2d = image.squeeze(2)
        images_flat = image_2d.view(image_2d.size(0), -1)
        flattened_images.append(images_flat)
    return torch.cat(flattened_images, dim=0)

In [5]:
class CatDataset(torch.utils.data.Dataset):
    def __init__(self, transform=None):
        trainset, _ = download_cifar10()
        cat_indices = [i for i, label in enumerate(trainset.targets) if label == 3]
        cat_subset = torch.utils.data.Subset(trainset, cat_indices)
        self.subset = cat_subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label 

In [6]:
from src.models.Autoencoder import ResNet50Encoder
from src.vmmd import VMMD

def init_vmmd():
    vmmd = VMMD(batch_size=100)
    return vmmd

def init_encoder():
    return ResNet50Encoder()

In [7]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor()
])
cat_dataset = CatDataset(transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [8]:
flattened_images = flatten_images_3d(cat_dataset)
flattened_images = flattened_images[:1000] #take only 1000 samples
print(flattened_images.shape)

torch.Size([1000, 3072])


In [9]:
resnet50 = init_encoder()

In [10]:
vmmd = init_vmmd()
vmmd.fit(cat_dataset, embedding_function=resnet50)

Epoch 0 of 30


Average loss in the epoch: 0.4576325416564941
Epoch 1 of 30


Average loss in the epoch: 0.4505611896514893
Epoch 2 of 30


Average loss in the epoch: 0.45197873115539555
Epoch 3 of 30


Average loss in the epoch: 0.4531123161315918
Epoch 4 of 30


Average loss in the epoch: 0.45678529739379875
Epoch 5 of 30


Average loss in the epoch: 0.4526765346527099
Epoch 6 of 30


Average loss in the epoch: 0.4465917587280273
Epoch 7 of 30


Average loss in the epoch: 0.4517050743103027
Epoch 8 of 30


KeyboardInterrupt: 